# Random forest: microbial composition to predict the babySQUID

Summary: Model using microbial composition to predict the babySQUID.

Author, date: Fannie Kerff, January-August 2025

Environment: qiime2-amplicon-2024.10 (saved in `environment.yml`)

## Setup

In [ ]:
# imports
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from functions_script import load_tsv, load_qza_artifact, prepare_features, evaluate_model

%matplotlib inline

In [ ]:
# paths
# IF ACCESS TO ALL STUDY DATA
path = "/cluster/work/bokulich/fkerff/GrumpyBiome-analysis"
meta_data_path = f"{path}/data-sensitive/meta_data"
seq_data_path = f"{path}/data-sensitive/processed_data"
figures_path = f"{path}/figures"

# ELSE (ONLY ACCESS TO PUBLIC DATA)
# meta_data_path = "../data/meta_data"
# figures_path = "../figures"

## Load

In [ ]:
# load samples metadata
md_samples = load_tsv(f"{meta_data_path}/md_samples.tsv")
md_samples = md_samples.sort_values(by=["timepoint", "infant_id"])

print(md_samples.shape)
md_samples.head()

In [ ]:
# load infant metadata per age/timepoint
md_ages = load_tsv(f"{meta_data_path}/md_ages.tsv")
md_ages = md_ages.sort_values(by=["timepoint", "infant_id"])

print(md_ages.shape)
md_ages.head()

In [ ]:
# Load ASVs relative abundances
asv_file_name = "rel-freq-feature-table"
feature_table = load_qza_artifact(f"{seq_data_path}/8-rel-freq-3035/{asv_file_name}.qza")

# Load Kmers relative abundances
kmer_table_filenames = ["rel-freq-kmer-table-10", "rel-freq-kmer-table-20", "rel-freq-kmer-table-50",
                        "rel-freq-tfidf-kmer-table-10", "rel-freq-tfidf-kmer-table-20", "rel-freq-tfidf-kmer-table-50"]
kmer_feature_tables = [load_qza_artifact(f"{seq_data_path}/7-kmer-results-3035/{filename}.qza") for filename in kmer_table_filenames]

## Prepare feature and label tables: ASV and K-mers

### ASVs

In [ ]:
# define data for babySQUID models
data_babySQUID = md_ages.dropna(subset=['babySQUID']).copy()

In [ ]:
# prepare feature and label arrays for model training by merging 
# feature data, metadata, and target sleep-quality measurements
X, y, groups, features = prepare_features(feature_table, md_samples, data_babySQUID)

### K-mers

In [ ]:
# similarly, prepare kmer feature and label sets
kmer_results = {}

for name, table in zip(kmer_table_filenames, kmer_feature_tables):
    kmer_results[name] = prepare_features(table, md_samples, data_babySQUID)

## Random Forest Classifier

### Split of babySQUID in good *versus* bad sleepers

In [ ]:
# compute babySQUID median
median_value = data_babySQUID['babySQUID'].median()

# define binary column: 0 if below median, 1 if >= median
data_babySQUID['good_sleepers'] = (data_babySQUID['babySQUID'] > median_value).astype(int)

In [ ]:
# visualize babySQUID distribution between good and bad sleepers
below_median = data_babySQUID[data_babySQUID['babySQUID'] <= median_value]['babySQUID']
above_median = data_babySQUID[data_babySQUID['babySQUID'] > median_value]['babySQUID']

print(data_babySQUID['good_sleepers'].value_counts())

plt.hist([below_median, above_median], 
         bins=20, 
         stacked=True, 
         color=['skyblue', 'salmon'], 
         label=['Below or Equal Median', 'Above Median'])

plt.axvline(median_value, color='gray', linestyle='dashed', linewidth=1.5, 
            label=f'Median = {median_value:.2f}')
plt.xlabel('babySQUID')
plt.ylabel('Frequency')
plt.title('Histogram of the babySQUID scores')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# prepare feature and label arrays for binary classification model training
X_bin, y_bin, groups_bin, features_bin = prepare_features(feature_table, md_samples, data_babySQUID, 'good_sleepers')

In [ ]:
# similarly, prepare kmer feature and label sets for binary classification
kmer_results_bin = {}

for name, table in zip(kmer_table_filenames, kmer_feature_tables):
    X, y, groups, features = prepare_features(table, md_samples, data_babySQUID, 'good_sleepers')
    kmer_results_bin[name] = (X, y, groups, features)

Perform train-test split based on infants then generate and train the RF model.

Evaluation on:
- Accuracy score: Accuracy = (Number of correct predictions) / (Total number of predictions)
- F1 Score: Harmonic mean of precision and recall (Precision: How many predicted positives are actual positives; Recall: How many actual positives the model identifies)

### ASVs

- **GroupShuffleSplit, GroupKFold and LeaveOneGroupOut**

In [ ]:
# RF binary classification model evaluation with GroupShuffleSplit cross-validation,
# GroupKFold cross-validation, and LeaveOneGroupOut cross-validation
cv = 'GroupShuffleSplit'
print(f"\n=== Evaluating model for: {asv_file_name} with {cv} cross-validation ===")
eval_asv_gss_bin = evaluate_model(X_bin, y_bin, groups_bin, train_test_split=cv, task_type="classification", n_splits=5, test_size=0.2)

cv = 'GroupKFold'
print(f"\n=== Evaluating model for: {asv_file_name} with {cv} cross-validation ===")
eval_asv_kfold_bin = evaluate_model(X_bin, y_bin, groups_bin, train_test_split=cv, task_type="classification", n_splits=5)

cv = 'LeaveOneGroupOut'
print(f"\n=== Evaluating model for: {asv_file_name} with {cv} cross-validation ===")
eval_asv_loo_bin = evaluate_model(X_bin, y_bin, groups_bin, train_test_split=cv, task_type="classification")


### Kmers

- **GroupShuffleSplit**

In [ ]:
# RF binary classification model evaluation for kmer features with GroupShuffleSplit cross-validation
cv = 'GroupShuffleSplit'
eval_kmer_gss_bin = {}

for name, (X, y, groups, features) in kmer_results_bin.items():
    print(f"\n=== Evaluating model for: {name} with {cv} cross-validation ===")

    eval_kmer_gss_bin[name] = evaluate_model(X, y, groups, train_test_split=cv, task_type="classification", n_splits=5, test_size=0.2)

- **GroupKFold**

In [ ]:
# RF binary classification model evaluation for kmer features with GroupKFold cross-validation
cv = 'GroupKFold'
eval_kmer_kfold_bin = {}

for name, (X, y, groups, features) in kmer_results_bin.items():
    print(f"\n=== Evaluating model for: {name} with {cv} cross-validation ===")

    eval_kmer_kfold_bin[name] = evaluate_model(X, y, groups, train_test_split=cv, task_type="classification", n_splits=5)

- **LOO approach**

In [ ]:
# RF binary classification model evaluation for kmer features with LeaveOneGroupOut cross-validation
cv = 'LeaveOneGroupOut'
eval_kmer_loo_bin = {}

for name, (X, y, groups, features) in kmer_results_bin.items():
    print(f"\n=== Evaluating model for: {name} with {cv} cross-validation ===")

    eval_kmer_loo_bin[name] = evaluate_model(X, y, groups, train_test_split=cv, task_type="classification", n_splits=5)

## Vizualizations of model performances

### Best model of **classifier** using rel-freq-tfidf-kmer-table-20

In [ ]:
# extract best model variables
model = eval_kmer_kfold_bin["rel-freq-tfidf-kmer-table-20"][0]
y_pred = eval_kmer_kfold_bin["rel-freq-tfidf-kmer-table-20"][1]
y_test = eval_kmer_kfold_bin["rel-freq-tfidf-kmer-table-20"][2]
X_test = eval_kmer_kfold_bin["rel-freq-tfidf-kmer-table-20"][3]

- **Confusion matrix**

Diagonal = correct predictions; Off-diagonal = misclassifications

In [ ]:
# compute confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# plot as heatmap
plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=model.classes_)
disp.plot()
plt.savefig(f"{figures_path}/confusion_matrix.pdf", dpi=300, bbox_inches='tight')
plt.show()